In [9]:
import pandas as pd
df = pd.read_csv("data/egypt_real_estate_listings.csv")
english = df[~df["description"].str.contains("[\u0600-\u06FF]", na=False)]
en_sample = english.sample(1000, random_state=42)
texts = en_sample["description"].dropna().tolist()
text = "\n".join(texts)


In [10]:
chars = sorted(set(text))
print(len(chars), chars)

127 ['\t', '\n', ' ', '!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '=', '>', '?', '@', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', '\\', ']', '_', '`', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '{', '|', '}', '~', '\xa0', '°', '²', '»', '¾', 'ç', 'é', '\u200b', '\u200e', '‑', '–', '—', '’', '“', '”', '•', '…', '\u202a', '\u202c', '\u202f', '\u2060', '€', '→', '▪', '▫', '◾', '✳', '⸻', '\uf0b7', 'ﬁ', '️', '\ufeff']


In [11]:
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}


In [13]:
import torch
data = torch.tensor([char_to_idx[ch] for ch in text])
data.shape

torch.Size([966938])

In [14]:
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

In [16]:
train_data.shape


torch.Size([870244])

In [17]:
val_data.shape

torch.Size([96694])

In [18]:
block_size = 64

In [21]:
ix = torch.randint(len(train_data)-block_size,(4,))

In [22]:
x = train_data[ix[0] : ix[0]+block_size]
y = train_data[ix[0]+1 : ix[0]+block_size+1]

In [24]:
batch_size = 4 
def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d)-block_size,(batch_size,))
    x = torch.stack([d[i:i+block_size]for i in ix])
    y = torch.stack([d[i+1:i+block_size+1]for i in ix])
    return x, y

In [41]:
x, y = get_batch('train')

In [36]:
x.shape

torch.Size([4, 64])

In [38]:
import torch.nn as nn
class TinyGPT(nn.Module):
    
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding= nn.Embedding(vocab_size, 64)
        self.head = nn.Linear(64,127)

    def forward(self, x):
        emb = self.embedding(x)
        logits = self.head(emb)
        return logits

In [39]:
model = TinyGPT(len(chars))
out = model(x)
print(out.shape)


torch.Size([4, 64, 127])


In [46]:
loss_fn = nn.CrossEntropyLoss()
loss = loss_fn(out.view(-1,127), y.view(-1))
print(loss.item())

5.042339324951172


In [56]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
for step in range(10000):
    x, y = get_batch('train')
    out = model(x)
    loss = loss_fn(out.view(-1,127),y.view(-1))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if step % 100 == 0:
        print(f"Step {step}, Loss: {loss.item():.4f}")


Step 0, Loss: 2.5117
Step 100, Loss: 2.4222
Step 200, Loss: 2.7628
Step 300, Loss: 2.5438
Step 400, Loss: 2.5651
Step 500, Loss: 2.5604
Step 600, Loss: 2.5484
Step 700, Loss: 2.6288
Step 800, Loss: 2.5952
Step 900, Loss: 2.0336
Step 1000, Loss: 2.6175
Step 1100, Loss: 2.8457
Step 1200, Loss: 2.6755
Step 1300, Loss: 2.0501
Step 1400, Loss: 2.6403
Step 1500, Loss: 2.4373
Step 1600, Loss: 2.4005
Step 1700, Loss: 2.8681
Step 1800, Loss: 2.5961
Step 1900, Loss: 2.5241
Step 2000, Loss: 2.5297
Step 2100, Loss: 2.5857
Step 2200, Loss: 2.6456
Step 2300, Loss: 2.2280
Step 2400, Loss: 2.5418
Step 2500, Loss: 2.7844
Step 2600, Loss: 2.1727
Step 2700, Loss: 2.5053
Step 2800, Loss: 2.1089
Step 2900, Loss: 2.1791
Step 3000, Loss: 2.3512
Step 3100, Loss: 2.4007
Step 3200, Loss: 2.5968
Step 3300, Loss: 2.5976
Step 3400, Loss: 2.5972
Step 3500, Loss: 2.5341
Step 3600, Loss: 2.5372
Step 3700, Loss: 2.4684
Step 3800, Loss: 2.5726
Step 3900, Loss: 2.6079
Step 4000, Loss: 2.5262
Step 4100, Loss: 2.8075
Step

In [57]:
def generate(model, start_char, length=200):
    idx = torch.tensor([char_to_idx[start_char]])
    result = [start_char]
    for _ in range(length):
        logits = model(idx.unsqueeze(0))
        probs = torch.softmax(logits[0, -1], dim=0)
        next_idx = torch.multinomial(probs, 1).item()
        result.append(idx_to_char[next_idx])
        idx = torch.cat([idx, torch.tensor([next_idx])])
    return ''.join(result)

print(generate(model, 'A'))


Alu ts eles, roul iede ( 172 Tatisrome squkil achita Ratit cl yom Zaw RAve Itee w sig :
Facloresed  Lorth ico Prhake ooff  ouste malay + ndieces fifre wa dit by oces, re aifon Bin Rew cules
51,00 bixp 
